# Mechanism transfer: an UNSEEN ligand class

**The question.** Three receptor representations that differ in what they know, tested on a
chemical class the graph **never saw during training**:

* **raw ESM** — *structure only*: sequence, sees no binding at all;
* **GNN+ESM** — *structure + function*: our graph, ESM node features refined by binding;
* **GNN one-hot** — *function only*: the same graph fed one-hot receptor identity, so the
  receptor geometry is learned entirely from binding, with zero structural prior.

If knowing structure *and* function beats either alone at an unseen mechanism, the two signals
are complementary; if one-hot already matches ESM-GNN, structure was not needed for this
transfer; if it collapses, structure is carrying the generalization.

**Design.** Pick a chemically coherent odorant class C (SMARTS), remove **every** class-C
odorant from training, train the full-coverage signed graph (q=0) on the rest, and read the
resulting receptor geometry several ways — three geometric measures of record (RSA, CCA,
Procrustes), a deprecated kNN panel, and a predictive OOD boosting head. `raw ESM` is the control: any refined-over-ESM gain is
binding-derived signal that generalised to an unseen chemistry.

**Which dataset is the stand.** The two insect matrices are complete — every receptor tested on
every odorant — so the measurement pattern is uniform and the target can be residualised against
each receptor's own tuning. M2OR is sparse and its assay pattern clusters by study, so a
receptor's "responds-to-C" label lives on a biased subset; it is kept as illustration, not as the
stand for the claim.

---

**This notebook computes nothing.** Everything below is produced by

```bash
.venv/bin/python scripts/modeling/analysis/mechanism_holdout.py --dataset all
```

and read from `results/mechanism_holdout/<dataset>/`. Set `DATASET` in the next cell.

In [ ]:
%matplotlib inline
import json, pathlib, sys
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import t as _t

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

DATASET = "cc"            # "m2or" | "cc" (Carey, mosquito) | "hc" (Hallem-Carlson, fly)
ART     = ROOT / "results/mechanism_holdout" / DATASET

meta = json.loads((ART / "meta.json").read_text())
res  = pd.read_csv(ART / "readouts.csv")
null = pd.read_csv(ART / "nulls.csv")
ood  = pd.read_csv(ART / "ood.csv") if (ART / "ood.csv").exists() else pd.DataFrame()
mols = pd.read_csv(ART / "molecules.csv")
panels   = np.load(ART / "panels.npz", allow_pickle=True)
overview = np.load(ART / "overview.npz", allow_pickle=True)

CLASSES = meta["classes"]
BINARY  = meta["kind"] == "sparse_binary"
OOD_METRIC = "AUROC" if meta["metric_family"] == "classification" else "R2"
KNN_LABEL  = "held-out class AUROC (LOO kNN)" if BINARY else "specificity (LOO kNN, tuning removed)"
KNN_BASE   = "tuning-only baseline" if BINARY else "permutation null"

MODELS  = ["raw ESM", "GNN+ESM full (q=0)", "GNN one-hot full (q=0)"]
PALETTE = {"raw ESM": "#888888", "GNN+ESM full (q=0)": "#2f5c9e",
           "GNN one-hot full (q=0)": "#b5651d",
           "receptor tuning": "#5aa469", "naive (train mean)": "#c0c0c0",
           "Hladis cls+mol": "#8a4f7d"}

def ci(x):
    x = np.asarray([v for v in np.asarray(x, float) if np.isfinite(v)])
    if len(x) < 2:
        return (float(x[0]) if len(x) else np.nan, 0.0)
    return x.mean(), _t.ppf(.975, len(x) - 1) * x.std(ddof=1) / len(x) ** .5

def agg(frame, cls, model, col):
    v = frame[(frame.cls == cls) & (frame.model == model)][col]
    return ci(v.to_numpy()) if len(v) else (np.nan, 0.0)

def table(frame, models, col, extra=None):
    rows = []
    for c in CLASSES:
        row = {"class": c}
        row.update({m: agg(frame, c, m, col)[0] for m in models})
        if extra is not None:
            row.update({k: float(null[null.cls == c][v].iloc[0]) for k, v in extra.items()})
        rows.append(row)
    df = pd.DataFrame(rows)
    display(df.style.format({c: "{:+.3f}" for c in df.columns if c != "class"}).hide(axis="index"))
    return df

def bars(frame, models, col, ylabel, title, lines=None, hline=None, ylim=None):
    fig, ax = plt.subplots(figsize=(2.0 * len(CLASSES) + 5, 4.6))
    x = np.arange(len(CLASSES)); w = 0.8 / len(models)
    for j, m in enumerate(models):
        vals = [agg(frame, c, m, col)[0] for c in CLASSES]
        errs = [agg(frame, c, m, col)[1] for c in CLASSES]
        ax.bar(x + (j - (len(models) - 1) / 2) * w, vals, w, yerr=errs, capsize=2,
               label=m, color=PALETTE.get(m))
    for name, (getter, style, colour) in (lines or {}).items():
        for xi, c in enumerate(CLASSES):
            ax.plot([xi - .45, xi + .45], [getter(c)] * 2, ls=style, c=colour, lw=1.3,
                    label=name if xi == 0 else None)
    if hline is not None:
        ax.axhline(hline, c="k", lw=.6)
    ax.set_xticks(x); ax.set_xticklabels(CLASSES, rotation=15, ha="right")
    ax.set_ylabel(ylabel); ax.set_title(f"{DATASET.upper()} — {title}")
    if ylim: ax.set_ylim(*ylim)
    ax.legend(frameon=False, ncol=3); fig.tight_layout(); plt.show()

print(f"{DATASET.upper()}: {meta['n_receptors']} receptors x {meta['n_odorants']} odorants, "
      f"{meta['n_pairs']} pairs | {meta['task']} | seeds {meta['seeds']} | "
      f"{meta['epochs']} epochs | computed in {meta['seconds']:.0f}s")
print("classes:", ", ".join(CLASSES), "| panel class:", meta["panel_class"])

## 1 · See the class

The response matrix with the held-out class pulled into a block on the left and the receptors
sorted by their response to it. If a distinct band of receptors lights up on the class block and
not elsewhere, the class is functionally atypical — which is what the `funcRedund` control
measures numerically, here by eye.

In [ ]:
M, n_class = overview["M"], int(overview["n_class"])
fig, ax = plt.subplots(figsize=(13, 5))
if BINARY:
    cmap = plt.cm.RdYlGn.copy(); cmap.set_bad("lightgray")
    im = ax.imshow(np.ma.masked_invalid(M), aspect="auto", cmap=cmap, vmin=0, vmax=1,
                   interpolation="nearest")
    cbar, note = "binds", "green = binds, red = no, grey = untested"
else:
    im = ax.imshow(M, aspect="auto", cmap="RdBu_r", vmin=-2, vmax=2, interpolation="nearest")
    cbar, note = "response (z)", "z-scored response"
ax.axvline(n_class - .5, color="black", lw=2.5)
ax.set_xlabel(f"odorants   [ {n_class} {overview['cls']} | {M.shape[1] - n_class} others ]")
ax.set_ylabel(f"{M.shape[0]} receptors (sorted by class response)")
ax.set_xticks([]); ax.set_yticks([])
ax.set_title(f"{DATASET.upper()} — {note}; class block left of the line")
fig.colorbar(im, fraction=.02, label=cbar); fig.tight_layout(); plt.show()

In [ ]:
from rdkit import Chem, RDLogger
from rdkit.Chem import Draw, AllChem, DataStructs
RDLogger.DisableLog("rdApp.*")

CVIZ = str(overview["cls"])
smi  = mols.set_index("inchikey")["smiles"].to_dict()
members = mols[mols[CVIZ]]["inchikey"].tolist()
others  = mols[~mols[CVIZ]]["inchikey"].tolist()

def fp(ik):
    m = Chem.MolFromSmiles(smi[ik])
    return AllChem.GetMorganFingerprintAsBitVect(m, 2, 2048) if m else None

def grid(iks, title, n=18):
    ms = [m for m in (Chem.MolFromSmiles(smi[i]) for i in iks[:n]) if m is not None]
    print(title); display(Draw.MolsToGridImage(ms, molsPerRow=6, subImgSize=(150, 110)))

fa   = [f for f in (fp(i) for i in members) if f is not None]
near = sorted(((i, max(DataStructs.BulkTanimotoSimilarity(fp(i), fa)))
               for i in others if fp(i) is not None), key=lambda t: -t[1])
rng  = np.random.default_rng(0)
grid(members, f"{CVIZ} — the held-out class ({len(members)} odorants); same functional hook")
grid([i for i, _ in near], "nearest NON-members (top Tanimoto to the class)")
grid(list(rng.choice(others, min(18, len(others)), replace=False)), "random odorants")

## 2 · The picture: actual class responses vs all three models

Each receptor's class-response block reconstructed from its neighbours in each embedding
(leave-one-out kNN). Four panels, one palette: **actual** | **GNN+ESM** (both) | **GNN one-hot**
(function only) | **ESM** (structure only). Panels that track the actual band carry the
mechanism; the gap between the two graph panels is what the structural prior adds on top of pure
binding.

On M2OR every model panel is **masked to `actual`'s measured support** — the matrix is sparse, so
this keeps each coloured cell paired with a ground truth and all four panels one shape. The insect
matrices are complete and need no mask.

In [ ]:
ro = panels["row_order"]
pset = [(panels["actual"], "actual"), (panels["gnn"], "GNN+ESM (both)"),
        (panels["onehot"], "GNN one-hot (function)"), (panels["esm"], "ESM (structure)")]
kw = dict(cmap="RdYlGn", vmin=0, vmax=1) if BINARY else dict(cmap="RdBu_r", vmin=-2, vmax=2)
if BINARY:
    kw["cmap"] = plt.cm.RdYlGn.copy(); kw["cmap"].set_bad("lightgray")
fig, axes = plt.subplots(1, 4, figsize=(19, 5.5), sharey=True)
for ax, (Mx, ttl) in zip(axes, pset):
    ax.imshow(np.ma.masked_invalid(Mx[ro]), aspect="auto", interpolation="nearest", **kw)
    ax.set_title(ttl, fontsize=10)
    ax.set_xlabel(f"{Mx.shape[1]} {panels['cls']}"); ax.set_xticks([]); ax.set_yticks([])
axes[0].set_ylabel(f"{len(ro)} receptors (sorted by class response)")
fig.suptitle(f"{DATASET.upper()} — {panels['cls']}: actual vs three models "
             f"(the class was never seen by any graph)")
fig.tight_layout(); plt.show()

## 3 · The kNN readout — *deprecated*

The original leave-one-out neighbour readout, kept so the panel still renders. **Do not quote
it**: `k`, the coarse receptor-level label and the smoothing are all moving parts nobody wanted
to defend, and the absolute height is inflated by design (receptors are all seen, the label is
coarse, the neighbour average smooths). Only the gap over ESM and the clearance over the
baseline ever carried meaning here.

In [ ]:
table(res, MODELS, "knn", extra={KNN_BASE: "knn_baseline"})
bars(res, MODELS, "knn", KNN_LABEL, f"kNN readout across classes — deprecated",
     lines={KNN_BASE: (lambda c: float(null[null.cls == c]["knn_baseline"].iloc[0]),
                       "--", "crimson")},
     hline=0.5 if BINARY else 0.0, ylim=(0.4, 1.0) if BINARY else None)

## 4 · The geometric readouts — **the metrics of record**

The same question with no moving parts at all: is the receptor geometry the graph produced
aligned with how those receptors respond to the class it never saw? No `k`, no smoothing, no
head, no hyperparameters — and nothing is predicted, which is exactly why these are the
trustworthy ones.

Three of them, deliberately of **increasing strictness**:

| | asks | invariant to |
|---|---|---|
| **RSA (Mantel)** | do close receptors in the embedding respond alike? | any monotone map of the similarities |
| **CCA** | is the class profile a linear function of the embedding at all? | any invertible linear map of either side |
| **Procrustes** | do the two clouds have the same *shape*? | rotation, reflection, uniform scale only |

A high CCA with a flat RSA would mean the information is present but not laid out
geometrically; Procrustes clearing its null is the strongest of the three claims. A
conclusion that survives all three does not depend on which notion of "aligned" one prefers.

Dotted line = row-permutation null (shuffle the embedding rows, the alignment dies).

**On the rank `k`.** CCA and Procrustes both need the two sides reduced to a common rank —
ESM is 1280-d over 24–50 receptors. That rank is a real knob, so it is calibrated rather than
guessed: CCA's null climbs with `k` and at `k=10` swallows the signal entirely (null 0.391,
raw ESM 0.425, the graph *below* the null), while Procrustes is insensitive to it. Hence the
small default; `meta.json` records the value actually used.

In [ ]:
GEOM = [g for g in ("rsa", "cca", "procrustes") if g in res.columns]
GEOM_LABEL = {"rsa": "RSA / Mantel (Spearman over off-diagonals)",
              "cca": "CCA (mean canonical correlation)",
              "procrustes": "Procrustes (1 - disparity)"}

print(f"geometry rank k = {meta.get('geometry_k_cap', '?')} "
      f"(per class, capped by n-1 and class size: {sorted(set(null['geometry_k']))})")
for g in GEOM:
    table(res, MODELS, g, extra={"null": f"{g}_null"})
    bars(res, MODELS, g, GEOM_LABEL[g], f"{g} across classes",
         lines={"permutation null": ((lambda gg: (lambda c: float(null[null.cls == c][f"{gg}_null"].iloc[0])))(g),
                                     ":", "gray")}, hline=0.0)

**Reading sections 3–4.** A class where a graph model stands clear of **both** the ESM bar **and**
its null — on all three geometric readouts — transferred something the graph never trained on. A class where everything sits at the
null is a real negative — only tuning lives there. The GNN+ESM vs one-hot gap says whether
sequence was needed on top of pure binding.

Read the insects with care on absolute height: 50 receptors (CC) and 24 (HC) mean strong
cross-correlation, so the ESM bar and the null — not the height — are what keep it fair.

## 5 · Predictive OOD: the pipeline's own boosting head

RSA deliberately predicts nothing. This puts prediction back with the smallest possible
apparatus: **no new model on top of the geometry**, just the boosting head the paper's tables
already use (`orbind.baselines.train_boost`, the same 400-tree XGBoost), fitted on
`[receptor vector ‖ molecule vector]` for every pair whose odorant is **outside** the held-out
class and scored on the class pairs. The receptor vector was itself built without the class, so
nothing in the path has seen that chemistry.

What this buys over RSA: it is a **controlled OOD**. The pipeline's cold-molecule split hides a
random 20% of odorants; here the hidden set is a chemical class, so "generalises to an unseen
molecule" sharpens into "generalises to an unseen **chemistry**". And unlike RSA it is a number
other methods can be scored on directly — section 6.

Two references on the same held-out pairs:

* **naive** — the training mean.
* **receptor tuning** — each receptor's mean response over the retained odorants: how much of the
  class is explained by "this receptor answers to everything".

In [ ]:
OOD_MODELS = [m for m in MODELS if m in set(ood["model"])] if len(ood) else []
REFS = [m for m in ["receptor tuning", "naive (train mean)"] if len(ood) and m in set(ood["model"])]

if not len(ood):
    print("no ood.csv — re-run the script without --no-ood")
else:
    table(ood, OOD_MODELS + REFS, OOD_METRIC)
    bars(ood, OOD_MODELS, OOD_METRIC, OOD_METRIC,
         f"predictive OOD on the held-out class ({OOD_METRIC})",
         lines={m: ((lambda mm: (lambda c: agg(ood, c, mm, OOD_METRIC)[0]))(m),
                    "--" if m == "receptor tuning" else ":", "gray") for m in REFS},
         hline=0.5 if meta["metric_family"] == "classification" else 0.0)

**Reading section 5.** Two questions, in order.

1. **Does anything clear the references?** Compare against the **higher** of `naive` and
   `receptor tuning` — under regression the tuning line is not automatically the weaker one: a
   receptor's mean over retained odorants can sit far from its class response, which drives its
   R² below naive. Whichever line is higher is the bar.
2. **Does the refined receptor beat raw ESM?** The same comparison RSA makes, now in units a
   competitor can also be scored in.

Agreement between the two readouts is the point: RSA has no head and no hyperparameters, this one
has both, so a conclusion surviving in both does not live in either one's moving parts.

Under regression, R² is measured against the **test** mean throughout. On a coherent class the
naive line is usually negative, and comparing an R² to 0 rather than to that line overstates every
model at once.

## 6 · The same OOD for a competitor

The point of section 5 is that this readout **transfers**: anything producing a pair-level
prediction can be dropped onto exactly these train/test masks. Below is Hladiš on those masks.

It is off by default in the script because it trains a model per (class, seed) rather than
reusing an embedding already computed — it is the expensive part, not a free one:

```bash
.venv/bin/python scripts/modeling/analysis/mechanism_holdout.py \
    --dataset cc --hladis --hladis-seeds 1
```

The budget follows the setting used elsewhere for each dataset (insects `2000:1200:100`,
M2OR `10000:6000:500`): upstream's own budget is sized for M2OR's 41k training rows and would
spend an insect run entirely inside the warm-up ramp.

In [ ]:
if len(ood) and "Hladis cls+mol" in set(ood["model"]):
    table(ood, OOD_MODELS + ["Hladis cls+mol"] + REFS, OOD_METRIC)
    bars(ood, OOD_MODELS + ["Hladis cls+mol"], OOD_METRIC, OOD_METRIC,
         f"predictive OOD incl. competitor ({OOD_METRIC})",
         lines={m: ((lambda mm: (lambda c: agg(ood, c, mm, OOD_METRIC)[0]))(m),
                    "--" if m == "receptor tuning" else ":", "gray") for m in REFS},
         hline=0.5 if meta["metric_family"] == "classification" else 0.0)
else:
    print("no competitor rows in ood.csv — re-run the script with --hladis")

## 7 — One number per representation: trust-weighted geometry

Sections 3–6 give a number per class. This one collapses them, so the three receptor
representations can be ranked outright. Two steps, both explicit.

**Dimensionless first.** Each class is scored against **its own** permutation null, in units
of that null's spread: `z = (value − null) / sd(null)`. Necessary, not cosmetic — the three
geometries sit on different floors (CCA's null is near 0.28, Procrustes' near 0.07), so raw
values cannot be averaged across measures, and a small class has a wider null than a large
one. `headroom = (value − null) / (1 − null)` is the same comparison in interpretable units:
the fraction of the distance from chance to a perfect match that was actually covered.

**Then weighted.** A class only tests mechanism transfer if the holdout truly isolated it,
and two leaks say it did not: `struct_leak` (mean best Tanimoto from a class member to a
retained odorant — a structural twin the graph did see) and `func_redund` (correlation
across receptors between the class target and general responsiveness — the class is just
tuning). `trust = (1 − struct_leak)(1 − func_redund)`, multiplicative because either leak
alone is enough to void the test. Both come from the artifacts, not from judgement.

The equal-weight column is printed beside the weighted one on purpose: if the two disagree,
the conclusion is a statement about which classes were isolated, and should be read as one.


In [ ]:
GEOM  = [g for g in ("rsa", "cca", "procrustes") if g in res.columns]
SHORT = {"raw ESM": "ESM", "GNN+ESM full (q=0)": "GNN+ESM", "GNN one-hot full (q=0)": "one-hot"}
nx    = null.set_index("cls")

def _null(c, g, col):
    return float(nx.loc[c, f"{g}_null{col}"]) if f"{g}_null{col}" in nx.columns else np.nan

def score(c, m, g, how):
    v, n0 = agg(res, c, m, g)[0], _null(c, g, "")
    if not np.isfinite(v) or not np.isfinite(n0):
        return np.nan
    if how == "headroom":
        return (v - n0) / (1.0 - n0)
    sd = _null(c, g, "_sd")
    return (v - n0) / sd if np.isfinite(sd) and sd > 1e-9 else np.nan

def wmean(vals, w):
    vals, w = np.asarray(vals, float), np.asarray(w, float)
    ok = np.isfinite(vals) & np.isfinite(w) & (w > 0)
    return float(np.sum(vals[ok] * w[ok]) / np.sum(w[ok])) if ok.any() else np.nan

TRUST = np.array([float(nx.loc[c, "trust"]) if "trust" in nx.columns else np.nan for c in CLASSES])
if not np.isfinite(TRUST).any():
    print("no trust column in nulls.csv — falling back to equal weights")
    TRUST = np.ones(len(CLASSES))
W = np.nan_to_num(TRUST) / np.nansum(TRUST)

display(pd.DataFrame({
    "class": CLASSES,
    "molecules": [int(nx.loc[c, "n_molecules"]) for c in CLASSES],
    "struct_leak": [float(nx.loc[c, "struct_leak"]) if "struct_leak" in nx.columns else np.nan
                    for c in CLASSES],
    "func_redund": [float(nx.loc[c, "func_redund"]) if "func_redund" in nx.columns else np.nan
                    for c in CLASSES],
    "trust": TRUST, "weight": W,
}).style.format({"struct_leak": "{:.3f}", "func_redund": "{:.3f}",
                 "trust": "{:.3f}", "weight": "{:.1%}"}).hide(axis="index"))

for how, fmt in (("z", "{:+.2f}"), ("headroom", "{:+.3f}")):
    rows = []
    for g in GEOM:
        row = {"measure": g}
        for m in MODELS:
            vals = [score(c, m, g, how) for c in CLASSES]
            row[SHORT[m]] = wmean(vals, W)
            row[f"{SHORT[m]} (equal)"] = wmean(vals, np.ones(len(CLASSES)))
        rows.append(row)
    df = pd.DataFrame(rows)
    print("")
    print(f"{how} — trust-weighted mean over {len(CLASSES)} classes, "
          f"beside the equal-weight mean")
    display(df.style.format({c: fmt for c in df.columns if c != "measure"}).hide(axis="index"))

fig, ax = plt.subplots(figsize=(1.9 * len(GEOM) + 5, 4.4))
x, w_ = np.arange(len(GEOM)), 0.8 / len(MODELS)
for j, m in enumerate(MODELS):
    ax.bar(x + (j - (len(MODELS) - 1) / 2) * w_,
           [wmean([score(c, m, g, "z") for c in CLASSES], W) for g in GEOM],
           w_, label=m, color=PALETTE.get(m))
ax.axhline(0, c="k", lw=.6)
ax.axhline(2, c="crimson", ls="--", lw=1.0, label="2 sd above the null")
ax.set_xticks(x); ax.set_xticklabels(GEOM)
ax.set_ylabel("trust-weighted z vs the permutation null")
ax.set_title(f"{DATASET.upper()} — one number per receptor representation")
ax.legend(frameon=False, ncol=2); fig.tight_layout(); plt.show()


**Reading section 7.** The bar is `0` — a representation at zero carries nothing about a
class it never saw. `2` is drawn as a reminder of scale, not as a test: the z is averaged over
classes, so it is an effect size, not a p-value, and nothing here corrects for the three
measures being three views of the same embeddings.

What the comparison is *for*: **one-hot** knows only binding, **raw ESM** only sequence, and
**GNN+ESM** both. If one-hot stands where ESM does not, the transferred mechanism came from
function rather than from structural similarity — the whole point of the holdout. If GNN+ESM
does not beat one-hot, ESM added nothing beyond what the graph already carried.

Weights make the statement conditional, and the condition is visible above: a class with high
`struct_leak` had structural twins left in training, and one with high `func_redund` was never
functionally distinct to begin with. Both are properties of the *dataset*, not of any model,
which is why they can be used as weights without favouring one representation.
